In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
%pip install -q onnx onnxruntime ultralytics 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 7.3 MB/s eta 0:00:00


In [5]:
from pathlib import Path
from time import perf_counter
import gc
import hashlib
import platform

import cv2
import numpy as np
import pandas as pd
import torch
import onnxruntime as ort
import ultralytics

from ultralytics import YOLO
from ultralytics.cfg import DEFAULT_CFG_DICT

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [ ]:
DRIVE = Path("/content/drive/MyDrive/Vision/Yolo_road2022_dataset/RDD2022_China_MotorBike")

MODEL_PT = DRIVE / "runs" / "yolo26n_rdd2022_baseline" / "weights" / "best.pt"
DATA_YAML =DRIVE / "configs" / "rdd2022_china_motorbike.yaml"
VAL_IMAGES = DRIVE / "val" / "images"

OUT = DRIVE / "out"
OUT.mkdir(exist_ok=True)

weight_hash = hashlib.sha256(MODEL_PT.read_bytes()).hexdigest()
print("Weights SHA256:", weight_hash)

Weights SHA256: f2bfe936f87b9b7362baa7e5318ed55b91271f42ed5af79e4262736daa5f9d99


In [7]:
extensions = {".jpg", ".jpeg", ".png", ".bmp"}

all_paths = sorted(
    [p for p in VAL_IMAGES.rglob("*") if p.suffix.lower() in extensions],
    key=lambda p: p.relative_to(VAL_IMAGES).as_posix(),
)

rng = np.random.default_rng(42)
indices = rng.choice(
    len(all_paths),
    size=min(50, len(all_paths)),
    replace=False,
)

sample_paths = [all_paths[int(i)] for i in indices]
images = [cv2.imread(str(p)) for p in sample_paths]

print("Latency images:", len(images))
print("First three:", [p.name for p in sample_paths[:3]])

Latency images: 50
First three: ['China_MotorBike_000787.jpg', 'China_MotorBike_000947.jpg', 'China_MotorBike_000856.jpg']


In [ ]:
IMGSZ = 640
WARMUP = 20
REPEATS = 3

COMMON = {
    "imgsz": IMGSZ,
    "rect": False,
    "iou": 0.70,
    "max_det": 300,
    "augment": False,
    "verbose": False,
}

def precision_args(fp16=False):
    if "quantize" in DEFAULT_CFG_DICT:
        return {"quantize": 16 if fp16 else 32}
    return {"half": fp16}


if DEFAULT_CFG_DICT.get("nms", False) is None:
    HEAD_ARGS = {"nms": False}
else:
    HEAD_ARGS = {"end2end": True}

print("FP32 arguments:", precision_args(False))
print("FP16 arguments:", precision_args(True))
print("Head arguments:", HEAD_ARGS)

FP32 arguments: {'quantize': 32}
FP16 arguments: {'quantize': 16}
Head arguments: {'nms': False}


**FP32 ONNX export**

In [10]:
export_args = {
    **HEAD_ARGS,
    **precision_args(False),
    "format": "onnx",
    "imgsz": IMGSZ,
    "batch": 1,
    "dynamic": False,
    "simplify": False,
    "opset": 17,
    "device": "cpu",
    "nms": False,
}

MODEL_ONNX = Path(
    YOLO(str(MODEL_PT)).export(**export_args)
).resolve()

assert MODEL_ONNX.is_file()
print("Exported:", MODEL_ONNX)

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n summary (fused): 122 layers, 2,375,616 parameters, 0 gradients, 5.3 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/Vision/Yolo_road2022_dataset/RDD2022_China_MotorBike/runs/yolo26n_rdd2022_baseline/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.1 MB)

ONNX: starting export with onnx 1.23.1 opset 17...
ONNX: export success ✅ 2.7s, saved as '/content/drive/MyDrive/Vision/Yolo_road2022_dataset/RDD2022_China_MotorBike/runs/yolo26n_rdd2022_baseline/weights/best.onnx' (9.3 MB)

Export complete (3.7s)
Results saved to /content/drive/MyDrive/Vision/Yolo_road2022_dataset/RDD2022_China_MotorBike/runs/yolo26n_rdd2022_baseline/weights/best.onnx
Predict:         yolo predict task=detect model=/content/drive/MyDrive/Vi

In [11]:
def synchronize(device):
    if device != "cpu":
        torch.cuda.synchronize()
        

def prepare_model(path, device="cpu", fp16=False):
    model = YOLO(str(path), task="detect")
    
    kwargs = {
        **COMMON,
        **HEAD_ARGS,
        **precision_args(fp16),
        "device": device,
        "conf": 0.25,
        "dnn": False,
        "save": False
    }
    
    for _ in range(WARMUP):
        model.predict(images[0], **kwargs)
    
    synchronize(device)
    backend = model.predictor.model
    print("Device:", backend.device, "| FP16:", backend.fp16)
    
    if Path(path).suffix == ".onnx":
        providers = backend.session.get_providers()
        print("Active providers:", providers)
        assert providers == ["CPUExecutionProvider"]

    return model, kwargs

**Measurement function**